<a href="https://colab.research.google.com/github/Arish3955/Route-Optimization-using-python/blob/main/routeoptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ortools folium


In [ ]:
!pip install -q ortools folium requests

In [ ]:
import math
import folium
from IPython.display import display
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# 1. DEFINE LOCATIONS (Index 0 is the Starting Depot)
locations = [
    {"name": "Depot - Civil Lines", "coords": (25.4520, 81.8338)},
    {"name": "Stop 1 - Prayagraj Junction", "coords": (25.4448, 81.8256)},
    {"name": "Stop 2 - Anand Bhawan", "coords": (25.4596, 81.8530)},
    {"name": "Stop 3 - Triveni Sangam", "coords": (25.4293, 81.8842)},
    {"name": "Stop 4 - Khusro Bagh", "coords": (25.4418, 81.8165)},
    {"name": "Stop 5 - Chandrashekhar Azad Park", "coords": (25.4526, 81.8447)},
    {"name": "Stop 6 - Naini Bridge", "coords": (25.4112, 81.8596)},
    {"name": "Stop 7 - Katra Market", "coords": (25.4645, 81.8481)}
]

# 2. COMPUTE DISTANCE MATRIX IN METERS
def haversine_meters(coord1, coord2):
    R = 6371000
    lat1 = math.radians(coord1[0])
    lon1 = math.radians(coord1[1])
    lat2 = math.radians(coord2[0])
    lon2 = math.radians(coord2[1])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return int(R * c)

def create_distance_matrix(locs):
    size = len(locs)
    matrix = []
    for i in range(size):
        row = []
        for j in range(size):
            if i == j:
                row.append(0)
            else:
                row.append(haversine_meters(locs[i]["coords"], locs[j]["coords"]))
        matrix.append(row)
    return matrix

# 3. SOLVE THE OPTIMAL ROUTE
def solve_tsp(distance_matrix):
    manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return distance_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    route = []
    if solution:
        index = routing.Start(0)
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        route.append(manager.IndexToNode(index))
        return route, solution.ObjectiveValue() / 1000.0
    return None, 0

dist_matrix = create_distance_matrix(locations)
optimal_route, total_km = solve_tsp(dist_matrix)

print("=== OPTIMAL DELIVERY ROUTE ===")
for step, idx in enumerate(optimal_route):
    print(f"Step {step}: {locations[idx]['name']}")
print(f"\nTotal Optimized Distance: {total_km:.2f} km\n")

# 4. RENDER MAP USING CARTODB TILES (UNBLOCKED IN COLAB)
start_coords = locations[0]["coords"]
route_map = folium.Map(
    location=start_coords,
    zoom_start=13,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(route_map)

all_coords = [loc["coords"] for loc in locations]
route_map.fit_bounds(all_coords)

for step, node_idx in enumerate(optimal_route[:-1]):
    loc = locations[node_idx]
    folium.Marker(
        location=loc["coords"],
        popup=f"Step {step}: {loc['name']}",
        tooltip=f"{step}. {loc['name']}",
        icon=folium.Icon(color="red" if node_idx == 0 else "blue", icon="home" if node_idx == 0 else "info-sign")
    ).add_to(route_map)

route_coords = [locations[idx]["coords"] for idx in optimal_route]
folium.PolyLine(
    locations=route_coords,
    color="green",
    weight=4,
    opacity=0.85,
    tooltip=f"Total Distance: {total_km:.2f} km"
).add_to(route_map)

route_map.save("optimized_route_map.html")
display(route_map)


=== OPTIMAL DELIVERY ROUTE ===
Step 0: Depot - Civil Lines
Step 1: Stop 5 - Chandrashekhar Azad Park
Step 2: Stop 7 - Katra Market
Step 3: Stop 2 - Anand Bhawan
Step 4: Stop 3 - Triveni Sangam
Step 5: Stop 6 - Naini Bridge
Step 6: Stop 4 - Khusro Bagh
Step 7: Stop 1 - Prayagraj Junction
Step 8: Depot - Civil Lines

Total Optimized Distance: 18.61 km

